# SpeechSense — управляющий запуск Kaggle
Сначала проверка данных и smoke. Полное обучение включается явно. GPU и Internet нужны для загрузки моделей. Test не используется при подборе.

In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT=Path("/kaggle/working/SpeechSense")
if not ROOT.exists(): subprocess.run(["git","clone","https://github.com/SamDerden/SpeechSense.git",str(ROOT)],check=True)
os.chdir(ROOT)
subprocess.run([sys.executable,"-m","pip","install","-r","requirements-kaggle.txt"],check=True)


После установки при необходимости перезапустите kernel. PyTorch используется из GPU-образа Kaggle; версия записывается в отчёт.

In [ ]:
import os,sys,subprocess
from pathlib import Path
ROOT=Path("/kaggle/working/SpeechSense")
os.chdir(ROOT)
files=list(Path("/kaggle/input").rglob("merged_sentiment_dataset.csv.gz"))
assert len(files)==1, f"Expected one dataset, found {files}"
DATA=files[0]
OUT=Path("/kaggle/working/speechsense_runs")
OUT.mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT/"src"))
from dataset import splits,verify_splits
import json,torch
assert torch.cuda.is_available(), "Enable GPU"
manifest=verify_splits(splits(DATA))
manifest["torch"]=torch.__version__
manifest["commit"]=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
(OUT/"manifest.json").write_text(json.dumps(manifest,indent=2))
print(manifest)


In [ ]:
RUN_FULL=False
subprocess.run([sys.executable,"src/run_suite.py","--data",str(DATA),"--out",str(OUT/("full" if RUN_FULL else "smoke")),*([] if RUN_FULL else ["--smoke"])],check=True)
